# M5-T8: Deep-Learning Stretch — Character-level CNN on Raw URLs

**Owner:** Sanjeewa Narayana  
**Depends on:** M4-T7 URL splits, M5-T3 (engineered-feature winner to beat)

**Run on GPU (Google Colab).** A character-level 1D CNN learns directly from the **raw URL string** (the `url` column), with no hand-engineered features. The question: can learning from raw characters beat the engineered-feature **Random Forest** from M5-T3?

**Bar to beat (M5-T3, engineered features):** Random Forest — **F1 = 0.866**, ROC-AUC = 0.965.

> Colab: `Runtime → Change runtime type → GPU` before running.

## Step 0 — Get the data
Download the M4-T7 URL splits from S3 (or upload them). Same files M5-T3 used, so the comparison is apples-to-apples.

In [2]:
# In Colab, fetch the same splits M5-T3 used (needs AWS creds) OR upload manually.
# !aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/splits/url_train.csv url_train.csv
# !aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/splits/url_test.csv  url_test.csv

import pandas as pd
from pathlib import Path

# Local repo path if running outside Colab; falls back to cwd in Colab
_base = Path('data/processed') if Path('data/processed/url_train.csv').exists() else Path('.')
tr = pd.read_csv(_base / 'url_train.csv')
te = pd.read_csv(_base / 'url_test.csv')
print('Train:', tr.shape, '| Test:', te.shape)
print('Label balance (train):', tr['label'].value_counts().to_dict())
tr['url'] = tr['url'].astype(str)
te['url'] = te['url'].astype(str)

Train: (512895, 13) | Test: (128224, 13)
Label balance (train): {0: 342464, 1: 170431}


## Step 1 — Character vocabulary + encoding
Build a char→int map from the training URLs and encode each URL to a fixed-length integer sequence (200 chars). Index 0 is reserved for padding / unknown chars.

In [3]:
import numpy as np

MAXLEN = 200
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Vocabulary from training URLs only (avoid test leakage)
chars = sorted(set(''.join(tr['url'].tolist())))
char2idx = {c: i + 1 for i, c in enumerate(chars)}   # 0 = pad/unknown
VOCAB = len(char2idx) + 1
print(f'Vocab size: {VOCAB}  (e.g. {chars[:20]})')

def encode(urls):
    out = np.zeros((len(urls), MAXLEN), dtype=np.int32)
    for i, u in enumerate(urls):
        for j, c in enumerate(u[:MAXLEN]):
            out[i, j] = char2idx.get(c, 0)
    return out

X_train = encode(tr['url'].tolist()); y_train = tr['label'].values
X_test  = encode(te['url'].tolist()); y_test  = te['label'].values
print('Encoded:', X_train.shape, X_test.shape)

Vocab size: 331  (e.g. ['\x01', '\x02', '\x03', '\x04', '\x05', '\x06', '\x07', '\x08', '\t', '\n', '\x0b', '\x0c', '\r', '\x0e', '\x0f', '\x10', '\x11', '\x12', '\x13', '\x14'])
Encoded: (512895, 200) (128224, 200)


## Step 2 — Char-CNN model
Embedding → parallel 1D convolutions (filter widths 3/4/5) → global max-pool → dense → sigmoid. Class weights handle the ~2:1 benign:malicious imbalance.

In [4]:
# If TensorFlow is missing, use a Python 3.12 conda env (TF is not available for Python 3.14):
# conda create -n tf312 python=3.12 -y
# conda activate tf312
# conda install -y -c conda-forge tensorflow
# Then restart kernel and re-run this notebook.

import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(RANDOM_STATE)

def build_char_cnn():
    inp = layers.Input(shape=(MAXLEN,), dtype='int32')
    x = layers.Embedding(VOCAB, 32, mask_zero=False)(inp)
    convs = []
    for k in (3, 4, 5):
        c = layers.Conv1D(128, k, activation='relu')(x)
        c = layers.GlobalMaxPooling1D()(c)
        convs.append(c)
    x = layers.Concatenate()(convs)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = Model(inp, out)
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return m

model = build_char_cnn()
model.summary()

cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight = {0: cw[0], 1: cw[1]}
print('class_weight:', class_weight)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 200, 32)   │     10,592 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 198, 128)  │     12,416 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 197, 128)  │     16,512 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 196, 128)  │     20,608 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d[0][0]      │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_1[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_2[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 384)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 384)       │          0 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │     24,640 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         65 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 84,833 (331.38 KB)

 Trainable params: 84,833 (331.38 KB)

 Non-trainable params: 0 (0.00 B)

class_weight: {0: np.float64(0.7488305340123341), 1: np.float64(1.504699849205837)}


## Step 3 — Train
A few epochs is plenty on ~513k URLs. Uses a small validation split for early visibility.

In [5]:
import time
t0 = time.time()
history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=512,
    class_weight=class_weight,
    verbose=1
)
train_time = time.time() - t0
print(f'\nTrained in {train_time:.1f}s')

Epoch 1/5
902/902 ━━━━━━━━━━━━━━━━━━━━ 107s 118ms/step - accuracy: 0.9428 - loss: 0.1612 - val_accuracy: 0.9719 - val_loss: 0.0860
Epoch 2/5
902/902 ━━━━━━━━━━━━━━━━━━━━ 123s 136ms/step - accuracy: 0.9685 - loss: 0.0968 - val_accuracy: 0.9771 - val_loss: 0.0688
Epoch 3/5
902/902 ━━━━━━━━━━━━━━━━━━━━ 132s 146ms/step - accuracy: 0.9737 - loss: 0.0815 - val_accuracy: 0.9799 - val_loss: 0.0613
Epoch 4/5
902/902 ━━━━━━━━━━━━━━━━━━━━ 135s 150ms/step - accuracy: 0.9765 - loss: 0.0723 - val_accuracy: 0.9810 - val_loss: 0.0577
Epoch 5/5
902/902 ━━━━━━━━━━━━━━━━━━━━ 145s 161ms/step - accuracy: 0.9789 - loss: 0.0656 - val_accuracy: 0.9814 - val_loss: 0.0567

Trained in 642.3s


## Step 4 — Evaluate (same metric set as the harness) + log
Compute the M5-T1 metric set on the held-out TEST split and append a row to `results/m5_results.csv` so it sits next to the M5-T3 models. (`cv_f1` is left blank — deep nets don't use the 5-fold CV the classical harness applies.)

In [6]:
import csv
from pathlib import Path
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

proba = model.predict(X_test, batch_size=1024).ravel()
pred = (proba >= 0.5).astype(int)

row = {
    'track': 'url',
    'model': 'Char-CNN (raw URLs)',
    'accuracy':  round(accuracy_score(y_test, pred), 4),
    'precision': round(precision_score(y_test, pred, zero_division=0), 4),
    'recall':    round(recall_score(y_test, pred, zero_division=0), 4),
    'f1':        round(f1_score(y_test, pred, zero_division=0), 4),
    'roc_auc':   round(roc_auc_score(y_test, proba), 4),
    'cv_f1':     '',
    'train_time': round(train_time, 2),
}
print(row)

# Append to the shared results log (create with header if missing)
RESULTS = Path('results/m5_results.csv') if Path('results').exists() else Path('m5_results.csv')
RESULTS.parent.mkdir(parents=True, exist_ok=True)
cols = ['track','model','accuracy','precision','recall','f1','roc_auc','cv_f1','train_time']
exists = RESULTS.exists()
with open(RESULTS, 'a', newline='') as f:
    w = csv.DictWriter(f, fieldnames=cols)
    if not exists: w.writeheader()
    w.writerow(row)

bar = 0.866  # M5-T3 Random Forest F1
verdict = 'BEATS' if row['f1'] > bar else 'does NOT beat'
print(f"\nChar-CNN F1 = {row['f1']}  vs  RandomForest (M5-T3) F1 = {bar}  ->  {verdict} the engineered-feature model.")

126/126 ━━━━━━━━━━━━━━━━━━━━ 9s 72ms/step
{'track': 'url', 'model': 'Char-CNN (raw URLs)', 'accuracy': 0.9826, 'precision': 0.9709, 'recall': 0.977, 'f1': 0.9739, 'roc_auc': 0.9976, 'cv_f1': '', 'train_time': 642.29}

Char-CNN F1 = 0.9739  vs  RandomForest (M5-T3) F1 = 0.866  ->  BEATS the engineered-feature model.


## Conclusion

- **Char-CNN test F1 = 0.9739** (ROC-AUC 0.9976, precision 0.9709, recall 0.9770) vs engineered-feature Random Forest **0.866**.
- **Verdict: the char-CNN decisively beats the engineered-feature model** — +0.108 F1 (0.974 vs 0.866) and +0.033 ROC-AUC. Learning directly from raw URL characters captures obfuscation patterns the 11 hand-crafted features miss, so the engineered features were clearly leaving signal on the table.
- **Cost:** ~642 s (~11 min) on GPU vs ~96 s for Random Forest on CPU — a very reasonable price for a large accuracy jump. **Recommendation: adopt the Char-CNN as the URL model for M6**, keeping Random Forest as a lightweight CPU fallback.
- **Note:** the CNN uses only the raw `url` string (no `url_length`, `num_dots`, etc.), so this is a genuinely different representation, not a re-run of M5-T3.
- **Logged:** row `Char-CNN (raw URLs)` appended to `results/m5_results.csv` (track=`url`).